# Classes, code-review-graph alone: guava (java)
Baseline. How many of the reference's classes does code-review-graph store as `Class` nodes?
Match = same file, name and start line. graph.db is opened read-only.

In [1]:
import pandas as pd, class_lab as L
pd.set_option('display.max_colwidth', 100); pd.set_option('display.width', 250)
APP, LANG = 'guava', 'java'
REPO = L.ROOT / 'apps' / APP
DB = L.ROOT / 'results' / APP / 'runs/iter-003_2026-11-17_2247/crg/graph.db'
files = L.repo_files(REPO, LANG)
print(LANG, 'files:', len(files))

java files: 3267


## 1. Reference: javac (`java-ref/ListClasses.java`)
Named classes are expected. Anonymous classes have no name and are not expected.

In [2]:
ref = L.reference(REPO, LANG, files)
print('parse errors:', ref.error.notna().sum())
ref = ref[ref.error.isna()]
print(ref.kind.value_counts().to_string())
named = ref[ref.kind != 'anonymous'].reset_index(drop=True)
print('expected named classes:', len(named))

parse errors: 0
kind
class         6416
anonymous     4444
interface      452
enum           392
annotation     140
expected named classes: 7400


## 2. code-review-graph Class nodes

In [3]:
crg = L.crg_classes(DB, REPO, LANG, files)
print('CRG Class nodes:', len(crg))

CRG Class nodes: 7260


## 3. Score

In [4]:
c, c_miss, c_extra = L.score(named, crg)
pd.Series(c).to_frame('code-review-graph')

,code-review-graph
expected,7400.0
found,7260.0
found %,98.1
same end line,7260.0
missed,140.0
extra,0.0


## 4. What failed
**Missed** = in the reference, no CRG node with the same file, name and start line.
**Extra** = CRG node with no match in the reference.

In [5]:
c_miss.assign(source=[L.source_line(REPO, f, l) for f, l in zip(c_miss.file, c_miss.line)])

,file,name,line,end,kind,source
37,android/guava-testlib/src/com/google/common/collect/testing/IgnoreJRERequirement.java,IgnoreJRERequirement,30,32,annotation,"@Target({METHOD, CONSTRUCTOR, TYPE, FIELD})"
119,android/guava-testlib/src/com/google/common/collect/testing/features/CollectionFeature.java,Require,118,125,annotation,@Retention(RetentionPolicy.RUNTIME)
121,android/guava-testlib/src/com/google/common/collect/testing/features/CollectionSize.java,Require,90,97,annotation,@Retention(RetentionPolicy.RUNTIME)
126,android/guava-testlib/src/com/google/common/collect/testing/features/ListFeature.java,Require,60,67,annotation,@Retention(RetentionPolicy.RUNTIME)
128,android/guava-testlib/src/com/google/common/collect/testing/features/MapFeature.java,Require,87,94,annotation,@Retention(RetentionPolicy.RUNTIME)
...,...,...,...,...,...,...
7143,guava/src/com/google/common/reflect/ParametricNullness.java,ParametricNullness,69,72,annotation,@GwtCompatible
7309,guava/src/com/google/common/util/concurrent/IgnoreJRERequirement.java,IgnoreJRERequirement,29,30,annotation,"@Target({METHOD, CONSTRUCTOR, TYPE, FIELD})"
7338,guava/src/com/google/common/util/concurrent/ParametricNullness.java,ParametricNullness,69,72,annotation,@GwtCompatible
7340,guava/src/com/google/common/util/concurrent/Partially.java,GwtIncompatible,41,46,annotation,@Retention(RetentionPolicy.CLASS)


In [6]:
c_extra.assign(source=[L.source_line(REPO, f, l) for f, l in zip(c_extra.file, c_extra.line_start)])

,file,name,line_start,line_end,source


## 5. False positives
**False positive** = a CRG Class node whose name is not a class in that file in the reference (any line).
A wrong start line is not a false positive: the class exists.

In [7]:
c_fp = L.false_positives(named, crg, REPO)
print('false positives (this language):', len(c_fp))
c_fp

false positives (this language): 0


,file,name,line_start,line_end,source


In [8]:
other = L.crg_other_files(DB, REPO, files)
print('CRG Class nodes in other file types:', len(other))
display(other.language.value_counts())
other.head(20)

CRG Class nodes in other file types: 0


Series([], Name: count, dtype: int64)

,file,name,line_start,language
